# RealSaS Mage Demo — FIT1 V5 + P1_B2_G10 exact binding / Run All

Demo-only product execution lane. **No Arachne FIT2 training. No barycentric skin transfer.**

1. Replays and materializes the already sealed `P1_B2_G10` V0..V7 meshes.
2. Loads sealed FIT1 V5 `K=4 x 512` field tokens plus the sealed 325,313-parameter DirectSimplex decoder.
3. Queries fresh weights on the **exact P1 mesh vertices**.
4. Compiler qualifies exact `M/G/W -> B` for every view.

This notebook does not claim mesh scientific PASS, PRODUCT_PASS or unseen generalization.


In [ ]:
# 0 — Drive + immutable source/input preflight
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, subprocess, shutil

SOURCE_COMMIT='2519714e7cf7de9cca8063521624ecfed4322dc0'
REPO=Path('/content/RealSaS-OPT')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','https://github.com/merynz/RealSaS-OPT.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',SOURCE_COMMIT],check=True)
head=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert head==SOURCE_COMMIT, (head,SOURCE_COMMIT)
os.chdir(REPO); sys.path.insert(0,str(REPO))

BASE=Path('/content/drive/MyDrive/REALSAS_MAGE_FULL_SUBJECT_RECLOSURE_20260912')
OBS=BASE/'IRIS_H1_V2_INPUT'/'20260912_FULL_SUBJECT'
ZERO=BASE/'IRIS_H1_V2_CONTINUATION_RUNS'/'20260912T074348Z'/'ZERO_SURFACE_PRODUCT_CLIPPED.npz'
A1=Path('/content/drive/MyDrive/ARACHNE_MAGE_A1_V4_FIT1_20260911/V5_EXACT_SKIN_REEMIT_V3_20260912')
EXACT=A1/'EXACT_PRODUCT_ARTIFACTS'
FIELD=EXACT/'MAGE_V5_EXACT_FIELD_TOKENS_GSA_FP32.npz'
WITNESS=EXACT/'MAGE_V5_EXACT_CONDITIONING_WITNESS.npz'
SKELETON=EXACT/'MAGE_FIT1_EXACT_QUALIFIED_SKELETON_IR.json'
SOURCE_SKIN=EXACT/'MAGE_V5_EXACT_QUALIFIED_SKIN_IR.json'
DECODER=A1/'ARACHNE_A1_V5_MINIMAL_K4_DIRECT_SIMPLEX_DECODER_DELTA_V1.pt'
required=[ZERO,FIELD,WITNESS,SKELETON,SOURCE_SKIN,DECODER]+[OBS/f'V{i}.png' for i in range(8)]+[OBS/f'V{i}.camera.json' for i in range(8)]
missing=[str(p) for p in required if not p.is_file()]
if missing: raise FileNotFoundError('MISSING_EXACT_INPUTS::'+json.dumps(missing))
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT=Path('/content/drive/MyDrive/REALSAS_MAGE_DEMO_FIT1_V5_P1_20260913')/RUN_ID
MESH_OUT=OUT/'P1_MATERIALIZED'; BIND_OUT=OUT/'V5_DIRECT_BINDING'
MESH_OUT.mkdir(parents=True,exist_ok=False); BIND_OUT.mkdir(parents=True,exist_ok=False)
print('PASS_SOURCE_AND_INPUT_PATH_PREFLIGHT', {'source':SOURCE_COMMIT,'out':str(OUT)})


In [ ]:
# 1 — Lightweight dependencies + repo contract tests
subprocess.run([sys.executable,'-m','pip','install','-q','numpy','scipy','pillow','pytest','scikit-image'],check=True)
subprocess.run([sys.executable,'-m','py_compile',
 'compiler/realsas_compiler_core/mesh/direct_model_skin.py',
 'experiments/mage_demo_fit1_v5_p1/materialize_p1_b2_g10_v1.py',
 'experiments/mage_demo_fit1_v5_p1/run_v5_direct_p1_binding_v1.py'],check=True)
subprocess.run([sys.executable,'-m','pytest','-q','tests/compiler/test_direct_model_mesh_skin_v1.py'],check=True)
print('PASS_LOCAL_PRODUCT_CONTRACT_PREFLIGHT')


In [ ]:
# 2 — Materialize exact sealed P1_B2_G10 meshes; lineage must match V0..V7 seal
cams=[str(OBS/f'V{i}.camera.json') for i in range(8)]
imgs=[str(OBS/f'V{i}.png') for i in range(8)]
cmd=[sys.executable,'experiments/mage_demo_fit1_v5_p1/materialize_p1_b2_g10_v1.py',
 '--zero-surface',str(ZERO),'--cameras',*cams,'--observations',*imgs,'--output-dir',str(MESH_OUT)]
subprocess.run(cmd,check=True)
mesh_seal=json.loads((MESH_OUT/'P1_B2_G10_MATERIALIZATION_SEAL.json').read_text())
assert mesh_seal['status']=='SEALED__EXACT_P1_B2_G10_V0_V7_MATERIALIZATION'
print('PASS_P1_MATERIALIZATION', mesh_seal['manifest_sha256'])


In [ ]:
# 3 — Fresh V5 W query on exact P1 vertices + Compiler exact M/G/W -> B qualification
device='cuda' if __import__('torch').cuda.is_available() else 'cpu'
cmd=[sys.executable,'experiments/mage_demo_fit1_v5_p1/run_v5_direct_p1_binding_v1.py',
 '--zero-surface',str(ZERO),'--cameras',*cams,'--observations',*imgs,
 '--materialized-mesh-dir',str(MESH_OUT),
 '--field-tokens',str(FIELD),'--conditioning-witness',str(WITNESS),'--decoder-delta',str(DECODER),
 '--skeleton',str(SKELETON),'--source-skin',str(SOURCE_SKIN),
 '--output-dir',str(BIND_OUT),'--device',device,'--chunk','2048']
subprocess.run(cmd,check=True)
bind_seal=json.loads((BIND_OUT/'V5_DIRECT_P1_BINDING_SEAL.json').read_text())
assert bind_seal['status']=='SEALED__V5_DIRECT_P1_BINDING'
print('PASS_V5_DIRECT_P1_BINDING', {'device':device,'manifest':bind_seal['manifest_sha256']})


In [ ]:
# 4 — Final fail-closed summary; still NOT PRODUCT_PASS
manifest=json.loads((BIND_OUT/'V5_DIRECT_P1_BINDING_MANIFEST.json').read_text())
assert manifest['status']=='PASS__V5_DIRECT_QUERY_AND_COMPILER_EXACT_P1_BINDING_V0_V7'
assert manifest['fit2_arachne_training_executed'] is False
assert manifest['historical_barycentric_weight_transfer_used'] is False
assert manifest['source_skin_rows_consumed'] is False
assert len(manifest['views'])==8
summary={
 'schema':'RealSaS.MageDemoFIT1V5P1NotebookSummary.v1',
 'status':'PASS__P1_MATERIALIZED__V5_DIRECT_W__COMPILER_EXACT_B',
 'source_commit':SOURCE_COMMIT,
 'run_id':RUN_ID,
 'mesh_materialization_manifest_sha256':mesh_seal['manifest_sha256'],
 'binding_manifest_sha256':bind_seal['manifest_sha256'],
 'fit2_arachne_training_executed':False,
 'product_pass_claimed':False,
 'next':'COMPONENT_ATTACHMENTS_THEN_PRODUCT_RESTORATION_FRAME0_DYNAMIC_PROOFS'
}
(OUT/'NOTEBOOK_RUN_SUMMARY.json').write_text(json.dumps(summary,indent=2,sort_keys=True)+'\n')
print(json.dumps(summary,indent=2))
print('TERMINAL_PASS__MAGE_DEMO_FIT1_V5_P1_DIRECT_BINDING_READY_FOR_PRODUCT_RESTORATION')
